# Skin Cancer Detection — ISIC 2024
## Notebook 5: LightGBM Model Training

**Goal:** Train a LightGBM classifier on the encoded dataset,
handling the extreme 1:1019 class imbalance using scale_pos_weight.

## Step 1 — Imports and Load Data

In [20]:
import pandas as pd
import numpy as np
import lightgbm as lgbm
from sklearn.model_selection import StratifiedKFold

In [21]:
data = pd.read_csv('/content/encoded_train_metadata.csv')

# Droping the redundant column from Week 3
data = data.drop(columns=['anatom_site_encoded'],errors='ignore')
print("Shape:", data.shape)

Shape: (401059, 39)


## Step 2 — Split X and y
## We separate features (X) from target (y) before building folds.

In [22]:
y = data['target']
X = data.drop(columns=['target'])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Cancer cases:", y.sum())
print("Cancer rate:", round(y.mean() * 100, 4), "%")


X shape: (401059, 38)
y shape: (401059,)
Cancer cases: 393
Cancer rate: 0.098 %


## Step 3 — Defineing LightGBM Parameters

In [23]:
params = {
    'objective': 'binary',
    'metric':'average_precision',
    'scale_pos_weight':1019,
    'n_estimators':1000,
    'learning_rate':0.05,
    'num_leaves':31,
    'random_state':42,
    'n_jobs':-1,
    'verbose':-1
}


print("Parameters set!")
print("scale_pos_weight:", params['scale_pos_weight'])


Parameters set!
scale_pos_weight: 1019


## Step 4 — Training Loop


In [24]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_prediction = np.zeros(len(X))

fold = 1

for train_idx,val_idx in skf.split(X,y):

  X_train, y_train = X.iloc[train_idx],y.iloc[train_idx]
  X_val,y_val = X.iloc[val_idx],y.iloc[val_idx]

  model = lgbm.LGBMClassifier(**params)
  model.fit(X_train,y_train)

  val_pred = model.predict_proba(X_val)[:,1]
  oof_prediction[val_idx] = val_pred

  print(f"Fold {fold} complete")
  fold += 1

print("All folds trained!")




Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
All folds trained!


In [25]:
print("OOF predictions shape:", oof_prediction.shape)
print("Min prediction:", round(oof_prediction.min(), 6))
print("Max prediction:", round(oof_prediction.max(), 6))
print("Mean prediction:", round(oof_prediction.mean(), 6))


OOF predictions shape: (401059,)
Min prediction: 0.0
Max prediction: 1.0
Mean prediction: 0.07519


## Summary

- Trained LightGBM with scale_pos_weight=1019
- Stored out-of-fold probability predictions for all 401,059 patients
- Ready for Week 6: Evaluating without accuracy metrics


In [26]:
np.save('oof_predictions.npy', oof_prediction)